# 📋 FILTER DATA REGISTRASI PESERTA SERTIFIKASI
## MCF (Azure AI-900) & MOS (Office 2019)

---
**Deskripsi Program:**
- Memfilter data peserta dari file `peserta mcf.csv` dan `peserta mos.csv`
- Cross-check dengan database Certiport (`certiport.csv`)
- Memberikan rekomendasi APPROVE / JANGAN APPROVE berdasarkan status ujian sebelumnya

**Input Files:**
1. `peserta mcf.csv` - Data pendaftar program MCF (Azure AI-900)
2. `peserta mos.csv` - Data pendaftar program MOS (Office 2019)
3. `certiport.csv` - Database hasil ujian dari Certiport

**Output Files:**
- `APPROVE_INI_belum_pernah_ujian.csv`
- `JANGAN_APPROVE_sudah_ujian.csv`
- `HASIL_FILTER_PESERTA_LENGKAP.xlsx`

---

## 1️⃣ Import Library & Setup

In [131]:
# ============================================================
# IMPORT LIBRARY YANG DIPERLUKAN
# ============================================================
import pandas as pd
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import warnings
warnings.filterwarnings('ignore')

print("✅ Library berhasil diimport!")
print("   - pandas: untuk manipulasi data")
print("   - fuzzywuzzy: untuk fuzzy string matching")

✅ Library berhasil diimport!
   - pandas: untuk manipulasi data
   - fuzzywuzzy: untuk fuzzy string matching


## 2️⃣ Load & Preview Data

In [132]:
# ============================================================
# LOAD DATA DARI FILE CSV (dari folder parent)
# ============================================================

# Load data peserta MCF dan MOS (dari folder parent)
peserta_mcf_df = pd.read_csv('../peserta mcf.csv')
peserta_mos_df = pd.read_csv('../peserta mos.csv')

# Load data Certiport (dari folder parent, skip 4 baris header)
certiport_df = pd.read_csv('../certiport.csv', skiprows=4)
certiport_df = certiport_df.dropna(axis=1, how='all')  # Hapus kolom kosong

# Tambahkan kolom sumber untuk identifikasi
peserta_mcf_df['Sumber'] = 'MCF'
peserta_mos_df['Sumber'] = 'MOS'

# Gabungkan kedua data peserta
peserta_df = pd.concat([peserta_mcf_df, peserta_mos_df], ignore_index=True)

# Filter data certiport berdasarkan jenis ujian
certiport_mcf = certiport_df[certiport_df['Exam'].str.contains('AI-900', na=False)].copy()
certiport_mos = certiport_df[certiport_df['Exam'].str.contains('Office 2019', na=False)].copy()

# ============================================================
# TAMPILKAN RINGKASAN DATA
# ============================================================
print(f"{'='*70}")
print("📊 RINGKASAN DATA YANG DILOAD")
print(f"{'='*70}")
print(f"\n📁 DATA PESERTA PENDAFTAR:")
print(f"   • Peserta MCF (Azure AI-900) : {len(peserta_mcf_df):>4} orang")
print(f"   • Peserta MOS (Office 2019)  : {len(peserta_mos_df):>4} orang")
print(f"   • TOTAL PESERTA              : {len(peserta_df):>4} orang")

print(f"\n📁 DATA CERTIPORT (HASIL UJIAN):")
print(f"   • Total Record               : {len(certiport_df):>4}")
print(f"   • Data MCF (AI-900)          : {len(certiport_mcf):>4}")
print(f"   • Data MOS (Office 2019)     : {len(certiport_mos):>4}")

print(f"\n📊 Breakdown Status Peserta:")
print(peserta_df['Status'].value_counts().to_string())

📊 RINGKASAN DATA YANG DILOAD

📁 DATA PESERTA PENDAFTAR:
   • Peserta MCF (Azure AI-900) :   91 orang
   • Peserta MOS (Office 2019)  :  146 orang
   • TOTAL PESERTA              :  237 orang

📁 DATA CERTIPORT (HASIL UJIAN):
   • Total Record               : 4853
   • Data MCF (AI-900)          :  897
   • Data MOS (Office 2019)     : 3445

📊 Breakdown Status Peserta:
Status
APPROVED        165
NOT APPROVED     72


## 3️⃣ Fungsi Fuzzy Matching

In [133]:
# ============================================================
# FUNGSI-FUNGSI UNTUK FUZZY MATCHING (IMPROVED v2)
# ============================================================

def normalize_name(name):
    """Normalisasi nama: uppercase, hapus karakter khusus kecuali - dan ., rapikan spasi"""
    if pd.isna(name):
        return ""
    # Keep alphanumeric, spaces, dash, and dot
    cleaned = ''.join(c for c in str(name) if c.isalnum() or c.isspace() or c in '.-')
    return ' '.join(cleaned.upper().strip().split())

def reverse_name(name):
    """Balik urutan nama (untuk antisipasi nama terbalik)"""
    parts = name.split()
    if len(parts) >= 2:
        return ' '.join(parts[::-1])
    return name

def count_matching_words(name1, name2, min_word_length=2):
    """Hitung jumlah kata yang sama antara dua nama"""
    words1 = set(w for w in name1.split() if len(w) >= min_word_length)
    words2 = set(w for w in name2.split() if len(w) >= min_word_length)
    return len(words1.intersection(words2)), len(words1), len(words2)

def get_min_matching_words(peserta_word_count, certiport_word_count):
    """
    Tentukan minimal kata yang harus cocok dengan aturan RELAXED:
    
    1. Jika Certiport punya ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
    2. Gunakan persentase 50% dari jumlah kata yang LEBIH SEDIKIT
    
    Contoh:
    - Form: 4 kata, Certiport: 2 kata → min(2, ceil(2*50%)) = min(2, 1) → 1 kata (tapi minimal 2)
    - Form: 4 kata, Certiport: 4 kata → ceil(4*50%) = 2 kata
    - Form: 3 kata, Certiport: 3 kata → ceil(3*50%) = 2 kata
    """
    import math
    
    # Ambil jumlah kata yang lebih sedikit
    min_words = min(peserta_word_count, certiport_word_count)
    
    # SPECIAL RULE: Jika Certiport hanya punya ≤2 kata, relax rule
    if certiport_word_count <= 2:
        # Minimal 2 kata cocok, atau semua kata certiport jika kurang dari 2
        return min(2, certiport_word_count)
    
    # Gunakan 50% dari kata yang lebih sedikit (minimal 2)
    percentage_required = math.ceil(min_words * 0.5)
    return max(2, percentage_required)

def is_single_word_match(name1, name2):
    """
    Untuk nama 1 kata, cek apakah ada tanda - atau . yang menunjukkan kecocokan
    Contoh: 'ABDUL-RAHMAN' atau 'A.RAHMAN'
    """
    # Cek apakah nama mengandung - atau .
    has_special_char = '-' in name1 or '.' in name1 or '-' in name2 or '.' in name2
    if not has_special_char:
        return False
    
    # Normalize tanpa - dan . untuk perbandingan
    clean1 = name1.replace('-', ' ').replace('.', ' ').upper()
    clean2 = name2.replace('-', ' ').replace('.', ' ').upper()
    
    # Cek kecocokan
    score = fuzz.token_set_ratio(clean1, clean2)
    return score >= 85

def find_best_match(peserta_name, certiport_names, threshold=90):
    """
    Mencari kecocokan nama terbaik dengan aturan RELAXED v2:
    - Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
    - Gunakan persentase 50% dari kata yang lebih sedikit
    - Minimal selalu 2 kata cocok (kecuali single word dengan - atau .)
    - Threshold dinaikkan ke 90 untuk menghindari false positive
    """
    peserta_normalized = normalize_name(peserta_name)
    peserta_words = [w for w in peserta_normalized.split() if len(w) >= 2]
    peserta_word_count = len(peserta_words)
    
    # Cek exact match
    if peserta_normalized in certiport_names:
        return peserta_normalized, 100, "Exact Match"
    
    # Cek nama terbalik
    reversed_name_str = reverse_name(peserta_normalized)
    if reversed_name_str in certiport_names:
        return reversed_name_str, 100, "Exact Match (Nama Terbalik)"
    
    best_score = 0
    best_match = None
    match_type = ""
    best_word_count = 0
    best_min_required = 0
    
    for cert_name in certiport_names:
        cert_words = [w for w in cert_name.split() if len(w) >= 2]
        cert_word_count = len(cert_words)
        
        # Special handling untuk nama 1 kata
        if peserta_word_count == 1:
            if is_single_word_match(peserta_normalized, cert_name):
                return cert_name, 90, "Single Word Match (dengan tanda - atau .)"
            continue
        
        matching_words, _, _ = count_matching_words(peserta_normalized, cert_name)
        
        # Hitung minimal kata yang harus cocok (RELAXED)
        min_words_required = get_min_matching_words(peserta_word_count, cert_word_count)
        
        # Cek minimum kata yang harus cocok
        if matching_words < min_words_required:
            continue
        
        # Berbagai metode fuzzy matching
        score1 = fuzz.ratio(peserta_normalized, cert_name)
        score2 = fuzz.token_set_ratio(peserta_normalized, cert_name)
        score3 = fuzz.token_sort_ratio(peserta_normalized, cert_name)
        score4 = fuzz.partial_ratio(peserta_normalized, cert_name)
        
        max_score = max(score1, score2, score3, score4)
        
        if matching_words > best_word_count or (matching_words == best_word_count and max_score > best_score):
            best_score = max_score
            best_match = cert_name
            best_word_count = matching_words
            best_min_required = min_words_required
            if max_score == score1:
                match_type = "Fuzzy Ratio"
            elif max_score == score2:
                match_type = "Token Set Ratio"
            elif max_score == score3:
                match_type = "Token Sort Ratio"
            else:
                match_type = "Partial Ratio"
    
    if best_score >= threshold and best_match:
        return best_match, best_score, f"{match_type} ({best_word_count}/{best_min_required} kata cocok)"
    
    return None, best_score, "Tidak Ditemukan"

print("✅ Fungsi matching (IMPROVED v2 - RELAXED) berhasil dibuat!")
print("\n📋 Aturan Matching BARU:")
print("   • Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)")
print("   • Gunakan persentase 50% dari kata yang LEBIH SEDIKIT")
print("   • Minimal selalu 2 kata cocok")
print("   • Nama 1 kata → harus ada tanda - atau . dan match")
print("\n📌 Contoh Aplikasi:")
print("   Form: 4 kata, Certiport: 2 kata → minimal 2 kata cocok ✓")
print("   Form: 4 kata, Certiport: 4 kata → minimal 2 kata cocok (50%)")
print("   Form: 3 kata, Certiport: 3 kata → minimal 2 kata cocok (50%)")

✅ Fungsi matching (IMPROVED v2 - RELAXED) berhasil dibuat!

📋 Aturan Matching BARU:
   • Jika Certiport ≤2 kata → minimal 2 kata cocok (atau semua kata certiport)
   • Gunakan persentase 50% dari kata yang LEBIH SEDIKIT
   • Minimal selalu 2 kata cocok
   • Nama 1 kata → harus ada tanda - atau . dan match

📌 Contoh Aplikasi:
   Form: 4 kata, Certiport: 2 kata → minimal 2 kata cocok ✓
   Form: 4 kata, Certiport: 4 kata → minimal 2 kata cocok (50%)
   Form: 3 kata, Certiport: 3 kata → minimal 2 kata cocok (50%)


## 4️⃣ Proses Cross-Check (Hanya NOT APPROVED)

In [134]:
# ============================================================
# PISAHKAN DATA BERDASARKAN STATUS
# ============================================================
print(f"{'='*70}")
print("📋 PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI")
print(f"{'='*70}")

approved_df = peserta_df[peserta_df['Status'] == 'APPROVED'].copy()
not_approved_df = peserta_df[peserta_df['Status'] == 'NOT APPROVED'].copy()

print(f"\n   ✅ Status APPROVED     : {len(approved_df):>4} orang")
print(f"   ⏳ Status NOT APPROVED : {len(not_approved_df):>4} orang")

# ============================================================
# PROSES CROSS-CHECK UNTUK NOT APPROVED
# ============================================================
print(f"\n{'='*70}")
print("🔍 PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED")
print(f"{'='*70}")

# Buat Full Name dan normalisasi untuk Certiport
certiport_mcf['Full Name'] = certiport_mcf['First Name'].fillna('') + ' ' + certiport_mcf['Last Name'].fillna('')
certiport_mos['Full Name'] = certiport_mos['First Name'].fillna('') + ' ' + certiport_mos['Last Name'].fillna('')

certiport_mcf_names = [normalize_name(name) for name in certiport_mcf['Full Name'].tolist()]
certiport_mos_names = [normalize_name(name) for name in certiport_mos['Full Name'].tolist()]

# Proses matching untuk NOT APPROVED
results = []
for idx, row in not_approved_df.iterrows():
    peserta_name = row['Nama']
    sumber_data = row['Sumber']
    
    # Pilih database certiport sesuai sumber
    certiport_names = certiport_mcf_names if sumber_data == 'MCF' else certiport_mos_names
    
    match, score, match_type = find_best_match(peserta_name, certiport_names)
    
    results.append({
        'Nama Peserta': peserta_name,
        'NIM': row['NIM'],
        'Jurusan': row['Jurusan'],
        'Sumber Data': sumber_data,
        'Program': row['Program Dipilih'],
        'Status Registrasi': row['Status'],
        'Nama di Certiport': match if match else '-',
        'Skor Kecocokan': score,
        'Tipe Match': match_type,
        'Status Certiport': '✅ DITEMUKAN' if match else '❌ TIDAK DITEMUKAN',
        'Rekomendasi': '❌ JANGAN APPROVE - Sudah Pernah Ujian' if match else '✅ APPROVE - Belum Pernah Ujian'
    })

results_df = pd.DataFrame(results)

print(f"\n✅ Proses cross-check selesai untuk {len(results)} peserta NOT APPROVED!")
print(f"\n📊 Hasil Cross-Check NOT APPROVED:")
print(results_df['Status Certiport'].value_counts().to_string())

# ============================================================
# PROSES VERIFIKASI UNTUK APPROVED (FILTER TAMBAHAN)
# ============================================================
print(f"\n{'='*70}")
print("🔍 VERIFIKASI PESERTA APPROVED - CEK APAKAH BENAR TIDAK ADA DI CERTIPORT")
print(f"{'='*70}")

approved_results = []
for idx, row in approved_df.iterrows():
    peserta_name = row['Nama']
    sumber_data = row['Sumber']
    
    # Pilih database certiport sesuai sumber
    certiport_names = certiport_mcf_names if sumber_data == 'MCF' else certiport_mos_names
    
    match, score, match_type = find_best_match(peserta_name, certiport_names)
    
    if match:
        status = '⚠️ PERLU DICEK - ADA DI CERTIPORT!'
        warning = True
    else:
        status = '✅ OK - Tidak ada di Certiport'
        warning = False
    
    approved_results.append({
        'Nama Peserta': peserta_name,
        'NIM': row['NIM'],
        'Jurusan': row['Jurusan'],
        'Sumber Data': sumber_data,
        'Program': row['Program Dipilih'],
        'Status Registrasi': row['Status'],
        'Nama di Certiport': match if match else '-',
        'Skor Kecocokan': score,
        'Tipe Match': match_type,
        'Status Verifikasi': status,
        'Perlu Dicek': warning
    })

approved_filter_df = pd.DataFrame(approved_results)
approved_warning_df = approved_filter_df[approved_filter_df['Perlu Dicek'] == True].copy()
approved_ok_df = approved_filter_df[approved_filter_df['Perlu Dicek'] == False].copy()

print(f"\n✅ Verifikasi APPROVED selesai!")
print(f"\n📊 Hasil Verifikasi APPROVED:")
print(f"   ✅ OK (Tidak ada di Certiport)      : {len(approved_ok_df):>4} peserta")
print(f"   ⚠️  PERLU DICEK (Ada di Certiport)  : {len(approved_warning_df):>4} peserta")

if len(approved_warning_df) > 0:
    print(f"\n⚠️ PERHATIAN! {len(approved_warning_df)} peserta APPROVED ternyata SUDAH ADA di Certiport!")
    print(f"   Peserta ini mungkin TIDAK SEHARUSNYA di-approve.")

📋 PEMISAHAN DATA BERDASARKAN STATUS REGISTRASI

   ✅ Status APPROVED     :  165 orang
   ⏳ Status NOT APPROVED :   72 orang

🔍 PROSES CROSS-CHECK UNTUK PESERTA NOT APPROVED

✅ Proses cross-check selesai untuk 72 peserta NOT APPROVED!

📊 Hasil Cross-Check NOT APPROVED:
Status Certiport
✅ DITEMUKAN          66
❌ TIDAK DITEMUKAN     6

🔍 VERIFIKASI PESERTA APPROVED - CEK APAKAH BENAR TIDAK ADA DI CERTIPORT

✅ Verifikasi APPROVED selesai!

📊 Hasil Verifikasi APPROVED:
   ✅ OK (Tidak ada di Certiport)      :  164 peserta
   ⚠️  PERLU DICEK (Ada di Certiport)  :    1 peserta

⚠️ PERHATIAN! 1 peserta APPROVED ternyata SUDAH ADA di Certiport!
   Peserta ini mungkin TIDAK SEHARUSNYA di-approve.


## 5️⃣ Hasil & Rekomendasi

In [135]:
# ============================================================
# PISAHKAN HASIL BERDASARKAN REKOMENDASI
# ============================================================

# Peserta yang ditemukan di Certiport (sudah pernah ujian)
found_df = results_df[results_df['Status Certiport'] == '✅ DITEMUKAN'].copy()

# Peserta yang tidak ditemukan di Certiport (belum pernah ujian)
not_found_df = results_df[results_df['Status Certiport'] == '❌ TIDAK DITEMUKAN'].copy()

# ============================================================
# TAMPILKAN HASIL
# ============================================================
print(f"{'='*80}")
print("✅ PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE (Belum Pernah Ujian)")
print(f"{'='*80}")
print(f"Total: {len(not_found_df)} peserta\n")
if len(not_found_df) > 0:
    print(not_found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Program', 'Rekomendasi']].to_string(index=False))

print(f"\n{'='*80}")
print("❌ PESERTA YANG JANGAN DI-APPROVE (Sudah Pernah Ujian)")
print(f"{'='*80}")
print(f"Total: {len(found_df)} peserta\n")
if len(found_df) > 0:
    print(found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan', 'Rekomendasi']].to_string(index=False))

✅ PESERTA YANG DIREKOMENDASIKAN UNTUK APPROVE (Belum Pernah Ujian)
Total: 6 peserta

                Nama Peserta       NIM Sumber Data           Program                    Rekomendasi
        UMMU PUTRI SALSABILA 202232036         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
MUHAMMAD FADHIL BIMA JULIANO 202231025         MCF MCF: Azure AI-900 ✅ APPROVE - Belum Pernah Ujian
   RADEN MUHAMMAD ASYAM RAFI 202211042         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
      MUHAMMAD DAFFA ZHAFRAN 202211115         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
MUHAMMAD WAHYU GUSTIAN HARUN 202214039         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian
           VIRGINIA KADIWARU 202331258         MOS         MOS: Word ✅ APPROVE - Belum Pernah Ujian

❌ PESERTA YANG JANGAN DI-APPROVE (Sudah Pernah Ujian)
Total: 66 peserta

                      Nama Peserta       NIM Sumber Data                  Nama di Certiport  Skor Kecocokan                           Rekomen

## 6️⃣ Deteksi Peserta Pernah Gagal

In [136]:
# ============================================================
# DETEKSI PESERTA YANG PERNAH GAGAL UJIAN
# ============================================================
print(f"{'='*70}")
print("⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN")
print(f"{'='*70}")

all_certiport = pd.concat([certiport_mcf, certiport_mos], ignore_index=True)
daftar_lagi_df = pd.DataFrame()

if 'Result' in all_certiport.columns:
    failed_students = all_certiport[all_certiport['Result'] == 'Fail'].copy()
    
    if len(failed_students) > 0:
        failed_students['Full Name Normalized'] = failed_students.apply(
            lambda x: normalize_name(f"{x['First Name']} {x['Last Name']}"), axis=1
        )
        
        not_approved_df_temp = not_approved_df.copy()
        not_approved_df_temp['Nama Normalized'] = not_approved_df_temp['Nama'].apply(normalize_name)
        
        daftar_lagi = []
        for idx, row in not_approved_df_temp.iterrows():
            nama_normalized = row['Nama Normalized']
            sumber = row['Sumber']
            
            if sumber == 'MCF':
                failed_to_check = failed_students[failed_students['Exam'].str.contains('AI-900', na=False)]
            else:
                failed_to_check = failed_students[failed_students['Exam'].str.contains('Office 2019', na=False)]
            
            for _, failed_row in failed_to_check.iterrows():
                failed_name = failed_row['Full Name Normalized']
                score = fuzz.token_set_ratio(nama_normalized, failed_name)
                if score >= 85 or nama_normalized == failed_name:
                    daftar_lagi.append({
                        'Nama Peserta': row['Nama'],
                        'NIM': row['NIM'],
                        'Sumber Data': sumber,
                        'Status': '⚠️ PERNAH GAGAL - DAFTAR LAGI',
                        'Nama di Certiport': f"{failed_row['First Name']} {failed_row['Last Name']}",
                        'Exam Sebelumnya': failed_row['Exam'],
                        'Skor Sebelumnya': failed_row.get('Score', '-'),
                        'Keputusan': '❓ PERLU REVIEW MANUAL'
                    })
                    break
        
        if len(daftar_lagi) > 0:
            daftar_lagi_df = pd.DataFrame(daftar_lagi)
            print(f"\n⚠️  PERHATIAN: {len(daftar_lagi_df)} peserta PERNAH GAGAL dan DAFTAR LAGI!")
            print(daftar_lagi_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Skor Sebelumnya', 'Keputusan']].to_string(index=False))
        else:
            print("\n✅ Tidak ada peserta NOT APPROVED yang pernah gagal ujian sebelumnya")
    else:
        print("\n✅ Tidak ada data peserta yang gagal di Certiport")
else:
    print("\n⚠️ Kolom 'Result' tidak ditemukan di data Certiport")

⚠️  DETEKSI PESERTA YANG PERNAH GAGAL UJIAN

⚠️  PERHATIAN: 61 peserta PERNAH GAGAL dan DAFTAR LAGI!
                      Nama Peserta       NIM Sumber Data  Skor Sebelumnya             Keputusan
              RAIHAN CANDRA IRAWAN 202211117         MOS            668.0 ❓ PERLU REVIEW MANUAL
                   MUHAMMAD RAIHAN 202241017         MOS            636.0 ❓ PERLU REVIEW MANUAL
MUHAMMAD REVIANSYAH DANENDRA PUTRA 202131160         MOS            152.0 ❓ PERLU REVIEW MANUAL
   DHEA HERAWATI INDAH PUTRI ERWIN 202231088         MOS            127.0 ❓ PERLU REVIEW MANUAL
               BINTAR ANDIKA PUTRA 202211113         MOS            547.0 ❓ PERLU REVIEW MANUAL
          RACHMAD FAQIH AFGIANSYAH 202112050         MOS            426.0 ❓ PERLU REVIEW MANUAL
                     JIHAN DIOVANT 202212037         MOS            578.0 ❓ PERLU REVIEW MANUAL
               NADELLA EZI SABRINA 202232010         MOS            147.0 ❓ PERLU REVIEW MANUAL
            JOHN ANDREW TAMPUBOLON 

## 7️⃣ Dashboard Summary

In [137]:
# ============================================================
# DASHBOARD SUMMARY
# ============================================================
print(f"\n{'='*90}")
print(" " * 30 + "📊 DASHBOARD SUMMARY")
print(f"{'='*90}")

total_peserta = len(peserta_df)
total_not_approved = len(not_approved_df)

print(f"""
┌────────────────────────────────────────────────────────────────────────────────┐
│                           DATA PENDAFTAR                                       │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar            : {total_peserta:>5} peserta                                   │
│    ├─ MCF (Azure AI-900)    : {len(peserta_mcf_df):>5} peserta ({len(peserta_mcf_df)/total_peserta*100:>5.1f}%)                    │
│    └─ MOS (Office 2019)     : {len(peserta_mos_df):>5} peserta ({len(peserta_mos_df)/total_peserta*100:>5.1f}%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                         STATUS REGISTRASI                                      │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED         : {len(approved_df):>5} peserta ({len(approved_df)/total_peserta*100:>5.1f}%)                    │
│  ⏳ Status NOT APPROVED     : {len(not_approved_df):>5} peserta ({len(not_approved_df)/total_peserta*100:>5.1f}%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                  HASIL CROSS-CHECK (NOT APPROVED)                              │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Ditemukan (Sudah Ujian) : {len(found_df):>5} peserta  → JANGAN APPROVE              │
│  ❌ Tidak Ditemukan         : {len(not_found_df):>5} peserta  → APPROVE INI                │
│  ⚠️  Pernah Gagal           : {len(daftar_lagi_df):>5} peserta  → PERLU REVIEW              │
├────────────────────────────────────────────────────────────────────────────────┤
│                  VERIFIKASI APPROVED (APPROVED FILTER)                         │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ OK (Tidak di Certiport) : {len(approved_ok_df):>5} peserta  → BENAR APPROVED           │
│  ⚠️  PERLU DICEK (Ada)      : {len(approved_warning_df):>5} peserta  → SEHARUSNYA TDK APPROVE   │
└────────────────────────────────────────────────────────────────────────────────┘
""")

print(f"\n🎯 REKOMENDASI AKHIR:")
print(f"   ✅ APPROVE     : {len(not_found_df)} peserta (belum pernah ujian)")
print(f"   ❌ JANGAN      : {len(found_df)} peserta (sudah pernah ujian)")
if len(daftar_lagi_df) > 0:
    print(f"   ⚠️ REVIEW     : {len(daftar_lagi_df)} peserta (pernah gagal)")
if len(approved_warning_df) > 0:
    print(f"   🔴 PERHATIAN  : {len(approved_warning_df)} peserta APPROVED tapi ADA di Certiport!")


                              📊 DASHBOARD SUMMARY

┌────────────────────────────────────────────────────────────────────────────────┐
│                           DATA PENDAFTAR                                       │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Pendaftar            :   237 peserta                                   │
│    ├─ MCF (Azure AI-900)    :    91 peserta ( 38.4%)                    │
│    └─ MOS (Office 2019)     :   146 peserta ( 61.6%)                    │
├────────────────────────────────────────────────────────────────────────────────┤
│                         STATUS REGISTRASI                                      │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Status APPROVED         :   165 peserta ( 69.6%)                    │
│  ⏳ Status NOT APPROVED     :    72 peserta ( 30.4%)                    │
├─────────────────────────────────────────────────────────────────

## 8️⃣ Export Hasil ke CSV

In [138]:
# ============================================================
# EXPORT HASIL KE FILE CSV
# ============================================================
print(f"{'='*70}")
print("💾 EXPORT HASIL KE FILE CSV")
print(f"{'='*70}")

# Export peserta yang direkomendasikan APPROVE
not_found_df.to_csv('APPROVE_INI_belum_pernah_ujian.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ APPROVE_INI_belum_pernah_ujian.csv")
print(f"   → {len(not_found_df)} peserta yang LAYAK DI-APPROVE")

# Export peserta yang JANGAN di-approve
found_df.to_csv('JANGAN_APPROVE_sudah_ujian.csv', index=False, encoding='utf-8-sig')
print(f"\n❌ JANGAN_APPROVE_sudah_ujian.csv")
print(f"   → {len(found_df)} peserta yang JANGAN DI-APPROVE")

# Export peserta APPROVED (langsung diterima)
approved_df.to_csv('peserta_approved.csv', index=False, encoding='utf-8-sig')
print(f"\n📋 peserta_approved.csv")
print(f"   → {len(approved_df)} peserta dengan status APPROVED")

# Export semua hasil cross-check
results_df.to_csv('hasil_crosscheck_not_approved.csv', index=False, encoding='utf-8-sig')
print(f"\n📊 hasil_crosscheck_not_approved.csv")
print(f"   → Semua hasil cross-check ({len(results_df)} peserta)")

# Export peserta yang perlu review
if len(daftar_lagi_df) > 0:
    daftar_lagi_df.to_csv('PERHATIAN_peserta_gagal_daftar_lagi.csv', index=False, encoding='utf-8-sig')
    print(f"\n⚠️ PERHATIAN_peserta_gagal_daftar_lagi.csv")
    print(f"   → {len(daftar_lagi_df)} peserta yang PERLU REVIEW")

💾 EXPORT HASIL KE FILE CSV

✅ APPROVE_INI_belum_pernah_ujian.csv
   → 6 peserta yang LAYAK DI-APPROVE

❌ JANGAN_APPROVE_sudah_ujian.csv
   → 66 peserta yang JANGAN DI-APPROVE

📋 peserta_approved.csv
   → 165 peserta dengan status APPROVED

📊 hasil_crosscheck_not_approved.csv
   → Semua hasil cross-check (72 peserta)

⚠️ PERHATIAN_peserta_gagal_daftar_lagi.csv
   → 61 peserta yang PERLU REVIEW


## 9️⃣ Export ke Excel dengan Smart Features

**FITUR BARU - EXPORT EXCEL PINTAR:**

File Excel yang dihasilkan sekarang memiliki **10 sheets** dengan fitur smart:

### 📊 **Sheets Standar (1-5):**
1. **Dashboard Summary** - Statistik keseluruhan
2. **APPROVE - Belum Ujian** - Peserta yang belum pernah ujian
3. **JANGAN APPROVE** - Peserta yang sudah ujian (🟡 kuning = skor <90)
4. **Approved Filter** - Verifikasi peserta APPROVED
5. **Semua Hasil Cross-Check** - Detail lengkap

### 🧠 **Sheets Smart - AUTO-REVIEW (6-10):**
6. **🟢 Auto-Approve (Confident)** - Peserta yang PASTI bisa di-approve
   - Skor rendah (<80) dan tidak ada di database manapun
   - Tidak perlu review manual lagi!

7. **🔴 Auto-Reject (Confident)** - Peserta yang PASTI ditolak
   - Skor tinggi (≥95) - sudah pasti match
   - Tidak perlu review manual!

8. **⚠️ Cross-DB Warning** - **PENTING!**
   - Peserta daftar di MCF tapi ditemukan di MOS (atau sebaliknya)
   - Perlu keputusan kebijakan

9. **🟡 Perlu Review Manual** - Focus di sini!
   - Kasus borderline (skor 80-94)
   - Minimalisir pengecekan manual

10. **🔍 Validation Issues** - Kualitas data
    - NIM duplikat, format nama aneh, dll

### 💡 **Keuntungan:**
- **Minimalkan review manual** - Fokus hanya ke sheet #9
- **Cross-database detection** - Deteksi peserta yang ujian program lain
- **Confident decisions** - Sheet #6 dan #7 sudah pasti
- **Data quality check** - Sheet #10 untuk cleanup

In [139]:
# ============================================================
# EXPORT KE EXCEL DENGAN FORMATTING + SMART FEATURES
# ============================================================
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print(f"{'='*70}")
print("📊 EXPORT KE EXCEL DENGAN SMART FEATURES")
print(f"{'='*70}")

# Jalankan auto-review untuk mendapatkan rekomendasi pintar
print("\n🤖 Menjalankan Auto-Review...")
review_results = auto_review()

# Jalankan validasi data
print("\n🔍 Menjalankan Validasi Data...")
validation_issues = validasi_data()

wb = Workbook()

# Style definitions
header_fill = PatternFill(start_color="1F4E79", end_color="1F4E79", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
approve_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")
reject_fill = PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
warning_fill = PatternFill(start_color="FFEB9C", end_color="FFEB9C", fill_type="solid")
manual_check_fill = PatternFill(start_color="FFFF00", end_color="FFFF00", fill_type="solid")  # Kuning untuk skor < 90
thin_border = Border(
    left=Side(style='thin'), right=Side(style='thin'),
    top=Side(style='thin'), bottom=Side(style='thin')
)

def style_worksheet(ws, highlight_col=None, approve_val=None, reject_val=None):
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center')
        cell.border = thin_border
    
    for row_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), start=2):
        for cell in row:
            cell.border = thin_border
        if highlight_col:
            cell_val = str(ws.cell(row=row_idx, column=highlight_col).value)
            fill = None
            if approve_val and approve_val in cell_val:
                fill = approve_fill
            elif reject_val and reject_val in cell_val:
                fill = reject_fill
            elif '⚠️' in cell_val or 'REVIEW' in cell_val:
                fill = warning_fill
            if fill:
                for cell in row:
                    cell.fill = fill
    
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 50)

# Sheet 1: Dashboard Summary
ws1 = wb.active
ws1.title = "Dashboard Summary"
summary = [
    ["KATEGORI", "JUMLAH", "PERSENTASE", "STATUS"],
    ["Total Pendaftar", len(peserta_df), "100%", "ℹ️"],
    ["  ├─ MCF (Azure)", len(peserta_mcf_df), f"{len(peserta_mcf_df)/len(peserta_df)*100:.1f}%", ""],
    ["  └─ MOS (Office)", len(peserta_mos_df), f"{len(peserta_mos_df)/len(peserta_df)*100:.1f}%", ""],
    ["", "", "", ""],
    ["Status APPROVED", len(approved_df), f"{len(approved_df)/len(peserta_df)*100:.1f}%", "✅"],
    ["Status NOT APPROVED", len(not_approved_df), f"{len(not_approved_df)/len(peserta_df)*100:.1f}%", "⏳"],
    ["", "", "", ""],
    ["Sudah Ujian (JANGAN APPROVE)", len(found_df), f"{len(found_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "❌"],
    ["Belum Ujian (APPROVE INI)", len(not_found_df), f"{len(not_found_df)/len(not_approved_df)*100:.1f}%" if len(not_approved_df) > 0 else "0%", "✅"],
]
for row in summary:
    ws1.append(row)
style_worksheet(ws1, 4, "✅", "❌")

# Sheet 2: APPROVE
ws2 = wb.create_sheet("APPROVE - Belum Ujian")
if len(not_found_df) > 0:
    for r in dataframe_to_rows(not_found_df[['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Program', 'Rekomendasi']], index=False, header=True):
        ws2.append(r)
    style_worksheet(ws2)
    for row in ws2.iter_rows(min_row=2, max_row=ws2.max_row):
        for cell in row:
            cell.fill = approve_fill

# Sheet 3: JANGAN APPROVE
ws3 = wb.create_sheet("JANGAN APPROVE - Sudah Ujian")
if len(found_df) > 0:
    for r in dataframe_to_rows(found_df[['Nama Peserta', 'NIM', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan', 'Rekomendasi']], index=False, header=True):
        ws3.append(r)
    style_worksheet(ws3)
    # Warnai berdasarkan Skor Kecocokan: Kuning jika < 90, Merah jika >= 90
    for row_idx, row in enumerate(ws3.iter_rows(min_row=2, max_row=ws3.max_row), start=2):
        skor_cell = ws3.cell(row=row_idx, column=5)  # Kolom Skor Kecocokan
        try:
            skor = float(skor_cell.value) if skor_cell.value else 100
            if skor < 90:
                # Skor rendah - perlu cek manual (kuning)
                for cell in row:
                    cell.fill = manual_check_fill
            else:
                # Skor tinggi - pasti sudah ujian (merah)
                for cell in row:
                    cell.fill = reject_fill
        except (ValueError, TypeError):
            # Jika skor tidak valid, gunakan warna merah default
            for cell in row:
                cell.fill = reject_fill

# Sheet 4: Approved Filter (VERIFIKASI PESERTA APPROVED)
ws4 = wb.create_sheet("Approved Filter")
if len(approved_filter_df) > 0:
    cols_to_show = ['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Program', 
                   'Nama di Certiport', 'Skor Kecocokan', 'Status Verifikasi']
    for r in dataframe_to_rows(approved_filter_df[cols_to_show], index=False, header=True):
        ws4.append(r)
    style_worksheet(ws4)
    # Apply coloring: Red for those found in Certiport (should not be approved), Green for OK
    for row_idx, row in enumerate(ws4.iter_rows(min_row=2, max_row=ws4.max_row), start=2):
        status_val = str(ws4.cell(row=row_idx, column=8).value)  # Status Verifikasi column
        if 'PERLU DICEK' in status_val or 'ADA DI CERTIPORT' in status_val:
            for cell in row:
                cell.fill = reject_fill  # Red fill for warning
        else:
            for cell in row:
                cell.fill = approve_fill  # Green fill for OK

# Sheet 5: Semua Hasil Cross-Check
ws5 = wb.create_sheet("Semua Hasil Cross-Check")
for r in dataframe_to_rows(results_df, index=False, header=True):
    ws5.append(r)
style_worksheet(ws5, 10, "TIDAK DITEMUKAN", "DITEMUKAN")

# ============================================================
# SHEET 6: AUTO-REVIEW - CONFIDENT APPROVE (SMART)
# ============================================================
ws6 = wb.create_sheet("🟢 Auto-Approve (Confident)")
if review_results['confident_approve']:
    df_auto_approve = pd.DataFrame(review_results['confident_approve'])
    for r in dataframe_to_rows(df_auto_approve, index=False, header=True):
        ws6.append(r)
    style_worksheet(ws6)
    # Warnai semua baris hijau
    for row in ws6.iter_rows(min_row=2, max_row=ws6.max_row):
        for cell in row:
            cell.fill = approve_fill
else:
    ws6.append(['Tidak ada peserta dengan confident approve'])

# ============================================================
# SHEET 7: AUTO-REVIEW - CONFIDENT REJECT (SMART)
# ============================================================
ws7 = wb.create_sheet("🔴 Auto-Reject (Confident)")
if review_results['confident_reject']:
    df_auto_reject = pd.DataFrame(review_results['confident_reject'])
    for r in dataframe_to_rows(df_auto_reject, index=False, header=True):
        ws7.append(r)
    style_worksheet(ws7)
    # Warnai semua baris merah
    for row in ws7.iter_rows(min_row=2, max_row=ws7.max_row):
        for cell in row:
            cell.fill = reject_fill
else:
    ws7.append(['Tidak ada peserta dengan confident reject'])

# ============================================================
# SHEET 8: CROSS-DATABASE WARNING (SMART)
# ============================================================
ws8 = wb.create_sheet("⚠️ Cross-DB Warning")
if review_results['cross_db_warning']:
    df_cross_db = pd.DataFrame(review_results['cross_db_warning'])
    for r in dataframe_to_rows(df_cross_db, index=False, header=True):
        ws8.append(r)
    style_worksheet(ws8)
    # Warnai orange (warning)
    for row in ws8.iter_rows(min_row=2, max_row=ws8.max_row):
        for cell in row:
            cell.fill = warning_fill
else:
    ws8.append(['Tidak ada cross-database warning'])

# ============================================================
# SHEET 9: PERLU REVIEW MANUAL (SMART)
# ============================================================
ws9 = wb.create_sheet("🟡 Perlu Review Manual")
if review_results['need_review']:
    df_need_review = pd.DataFrame(review_results['need_review'])
    for r in dataframe_to_rows(df_need_review, index=False, header=True):
        ws9.append(r)
    style_worksheet(ws9)
    # Warnai kuning
    for row in ws9.iter_rows(min_row=2, max_row=ws9.max_row):
        for cell in row:
            cell.fill = manual_check_fill
else:
    ws9.append(['Tidak ada peserta yang perlu review manual'])

# ============================================================
# SHEET 10: DATA VALIDATION ISSUES
# ============================================================
ws10 = wb.create_sheet("🔍 Validation Issues")
if validation_issues:
    ws10.append(['Issue Type', 'Description'])
    for issue in validation_issues:
        ws10.append(['Data Quality', issue])
    style_worksheet(ws10)
else:
    ws10.append(['Data Quality', 'Tidak ada issue'])
    style_worksheet(ws10)

# Simpan file
excel_file = 'HASIL_FILTER_PESERTA_LENGKAP.xlsx'
wb.save(excel_file)

print(f"\n{'='*70}")
print(f"✅ File Excel SMART berhasil disimpan: {excel_file}")
print(f"{'='*70}")
print(f"\n📋 DAFTAR SHEETS (Total: 10 sheets):")
print(f"\n   📊 SHEETS STANDAR:")
print(f"   1. Dashboard Summary          - Ringkasan statistik keseluruhan")
print(f"   2. APPROVE - Belum Ujian      - 🟢 {len(not_found_df)} peserta (standar)")
print(f"   3. JANGAN APPROVE - Sudah     - 🔴 {len(found_df)} peserta (🟡 skor <90)")
print(f"   4. Approved Filter            - 🔍 Verifikasi {len(approved_filter_df)} APPROVED")
print(f"   5. Semua Hasil Cross-Check    - Detail lengkap semua peserta")
print(f"\n   🧠 SHEETS SMART (BARU):")
print(f"   6. 🟢 Auto-Approve            - {len(review_results['confident_approve'])} peserta PASTI APPROVE")
print(f"   7. 🔴 Auto-Reject             - {len(review_results['confident_reject'])} peserta PASTI REJECT")
print(f"   8. ⚠️ Cross-DB Warning        - {len(review_results['cross_db_warning'])} peserta ada di program lain")
print(f"   9. 🟡 Perlu Review Manual     - {len(review_results['need_review'])} peserta perlu dicek")
print(f"  10. 🔍 Validation Issues       - {len(validation_issues)} data quality issues")
print(f"\n{'='*70}")
print(f"💡 REKOMENDASI:")
print(f"   ✅ Fokus review {len(review_results['need_review'])} peserta di sheet #9")
print(f"   ✅ Gunakan sheet #6 dan #7 untuk keputusan confident")
print(f"   ⚠️  Perhatikan sheet #8 untuk cross-database cases")
print(f"{'='*70}")

📊 EXPORT KE EXCEL DENGAN SMART FEATURES

🤖 Menjalankan Auto-Review...

🤖 AUTO-REVIEW - ANALISIS OTOMATIS SEMUA HASIL

Menganalisis 72 peserta NOT APPROVED...

✅ CONFIDENT APPROVE (1 peserta)
             Nama  Skor                                   Reason
VIRGINIA KADIWARU    31 Skor rendah (31%), tidak ada di semua DB

❌ CONFIDENT REJECT (66 peserta)
                              Nama  Skor                              Match                          Reason
         MUHAMMAD RIFQI APRIANSYAH   100                     MUHAMMAD RIFQI Skor tinggi (100%), pasti match
              RAIHAN CANDRA IRAWAN   100                      RAIHAN IRAWAN Skor tinggi (100%), pasti match
                   MUHAMMAD RAIHAN   100                    MUHAMMAD RAIHAN Skor tinggi (100%), pasti match
MUHAMMAD REVIANSYAH DANENDRA PUTRA   100 MUHAMMAD REVIANSYAH DANENDRA PUTRA Skor tinggi (100%), pasti match
   DHEA HERAWATI INDAH PUTRI ERWIN   100          DHEA HERAWATI PUTRI ERWIN Skor tinggi (100%), pasti matc

   ⚠️  Ditemukan 24 NIM duplikat!
      NIM 202131047: 2x
         - VIVIN MELANESYA WAYENI (MCF)
         - VIVIN MELANESYA WAYENI (MOS)
      NIM 202131080: 2x
         - AULIA PRAVDA SULISTYASEVA (MCF)
         - AULIA PRAVDA SULISTYASEVA (MOS)
      NIM 202131124: 2x
         - JOVAN DRA (MCF)
         - JOVAN DRA (MOS)
      NIM 202131160: 2x
         - MUHAMMAD REVIANSYAH DANENDRA PUTRA (MCF)
         - MUHAMMAD REVIANSYAH DANENDRA PUTRA (MOS)
      NIM 202231008: 2x
         - NADYA AURA SALZABILA RAMADHANI (MCF)
         - NADYA AURA SALZABILA RAMADHANI (MOS)
      NIM 202231009: 2x
         - LALU MUHAMMAD RISGAN NAZWA (MCF)
         - LALU MUHAMMAD RISGAN NAZWA (MOS)
      NIM 202231012: 2x
         - YERICHO HARVEY KRISETYANTO (MCF)
         - YERICHO HARVEY KRISETYANTO (MOS)
      NIM 202231015: 2x
         - MUHAMMAD AKBAR ABIDZAR AL GHIFARI (MCF)
         - MUHAMMAD AKBAR ABIDZAR AL GHIFARI (MOS)
      NIM 202231020: 2x
         - DIMAS DAMARJATI (MCF)
         - DIMAS DA

## 📈 Ringkasan Efisiensi Smart Tools

Dengan fitur smart tools yang baru, Anda dapat:
- ✅ **Mengurangi pengecekan manual hingga 80-90%**
- ✅ **Deteksi otomatis cross-database cases**
- ✅ **Identifikasi data quality issues**
- ✅ **Confident decisions tanpa review manual**

In [140]:
# ============================================================
# 📊 VISUALISASI EFISIENSI SMART TOOLS
# ============================================================

# Hitung statistik efisiensi
total_peserta_not_approved = len(results_df)
confident_decisions = len(review_results['confident_approve']) + len(review_results['confident_reject'])
need_manual_review = len(review_results['need_review']) + len(review_results['cross_db_warning'])

efficiency_rate = (confident_decisions / total_peserta_not_approved * 100) if total_peserta_not_approved > 0 else 0

print(f"\n{'='*80}")
print(f"📊 EFISIENSI SMART TOOLS")
print(f"{'='*80}")
print(f"""
┌────────────────────────────────────────────────────────────────────────────────┐
│                         SEBELUM SMART TOOLS                                    │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Peserta NOT APPROVED     : {total_peserta_not_approved:>4} peserta                               │
│  Perlu Review Manual            : {total_peserta_not_approved:>4} peserta (100.0%)                      │
│  Waktu Review (estimasi)        : {total_peserta_not_approved * 2:>4} menit (@2 menit/peserta)          │
├────────────────────────────────────────────────────────────────────────────────┤
│                         SESUDAH SMART TOOLS                                    │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Confident Approve           : {len(review_results['confident_approve']):>4} peserta                               │
│  ❌ Confident Reject            : {len(review_results['confident_reject']):>4} peserta                               │
│  🟡 Perlu Review Manual         : {need_manual_review:>4} peserta ({need_manual_review/total_peserta_not_approved*100:>5.1f}%)                      │
│  ⚠️  Cross-DB Warning           : {len(review_results['cross_db_warning']):>4} peserta                               │
├────────────────────────────────────────────────────────────────────────────────┤
│                              PENGHEMATAN                                       │
├────────────────────────────────────────────────────────────────────────────────┤
│  Otomatis Terproses             : {confident_decisions:>4} peserta ({efficiency_rate:>5.1f}%)                      │
│  Waktu Review Baru (estimasi)   : {need_manual_review * 2:>4} menit (@2 menit/peserta)          │
│  Waktu Dihemat                  : {(total_peserta_not_approved - need_manual_review) * 2:>4} menit                               │
│                                                                                │
│  🎯 EFISIENSI: {efficiency_rate:>5.1f}% keputusan otomatis!                              │
└────────────────────────────────────────────────────────────────────────────────┘
""")

print(f"💡 KESIMPULAN:")
print(f"   Anda hanya perlu review {need_manual_review} dari {total_peserta_not_approved} peserta!")
print(f"   Penghematan waktu: ~{(total_peserta_not_approved - need_manual_review) * 2} menit")
print(f"   Tingkat akurasi: Tinggi (threshold 90, cross-DB check)")
print(f"{'='*80}")


📊 EFISIENSI SMART TOOLS

┌────────────────────────────────────────────────────────────────────────────────┐
│                         SEBELUM SMART TOOLS                                    │
├────────────────────────────────────────────────────────────────────────────────┤
│  Total Peserta NOT APPROVED     :   72 peserta                               │
│  Perlu Review Manual            :   72 peserta (100.0%)                      │
│  Waktu Review (estimasi)        :  144 menit (@2 menit/peserta)          │
├────────────────────────────────────────────────────────────────────────────────┤
│                         SESUDAH SMART TOOLS                                    │
├────────────────────────────────────────────────────────────────────────────────┤
│  ✅ Confident Approve           :    1 peserta                               │
│  ❌ Confident Reject            :   66 peserta                               │
│  🟡 Perlu Review Manual         :    5 peserta (  6.9%)                     

---

## 🎉 PROGRAM SELESAI - SEMUA FITUR SMART AKTIF!

### ✅ **Fitur yang Tersedia:**

#### 1️⃣ **Matching System (RELAXED v2)**
- Threshold: 90 (menghindari false positive)
- Dynamic word matching (50% rule)
- Special handling untuk nama pendek

#### 2️⃣ **Smart Search Tools**
- `cari_nama()` - Cari di database dengan detail
- `cari_nama_mirip()` - Tampilkan kandidat mirip
- `cari_semua_database()` - Cross-database search
- `cek_batch()` - Batch processing

#### 3️⃣ **Smart Filters**
- `cari_peserta()` - Multi-criteria search
- `filter_borderline()` - Kasus borderline
- `filter_high_confidence()` - Match tinggi

#### 4️⃣ **Smart Analysis**
- `validasi_data()` - Data quality check
- `analisis_distribusi_skor()` - Score distribution
- `auto_review()` - **Auto-categorization dengan cross-DB check**

#### 5️⃣ **Smart Export**
- **10 sheets Excel** dengan auto-categorization
- Color coding berdasarkan confidence level
- Cross-database warning detection
- Data validation issues

#### 6️⃣ **Utilities**
- `status_cepat()` - Quick status check
- `export_filtered()` - Custom export
- `bantuan()` - Help guide

### 🎯 **Cara Penggunaan:**
1. **Jalankan semua cell dari atas ke bawah**
2. **Cek hasil di file Excel** `HASIL_FILTER_PESERTA_LENGKAP.xlsx`
3. **Fokus review di sheet #9** (Perlu Review Manual)
4. **Gunakan fungsi search** untuk cek nama spesifik
5. **Export custom** dengan `export_filtered()` jika perlu

### 📞 **Quick Commands:**
```python
# Cek nama cepat
status_cepat('NAMA')

# Lihat nama mirip
cari_nama_mirip('NAMA', 'SEMUA', 10)

# Cari multi-kriteria
cari_peserta(jurusan='Informatika', skor_min=85)

# Bantuan lengkap
bantuan()
```

---

## 🔍 Fungsi Pencarian Manual

## 🧠 Smart Tools - Fitur Pencarian & Filter Lanjutan

Kumpulan fungsi pintar untuk memudahkan pencarian dan analisis data:
- **Pencarian Multi-Kriteria** - Cari berdasarkan nama, NIM, jurusan, skor
- **Batch Lookup** - Cek banyak nama sekaligus
- **Similar Names Finder** - Temukan nama mirip di Certiport
- **Cross-Database Search** - Cek di MCF & MOS sekaligus
- **Data Validation** - Deteksi data bermasalah
- **Quick Filters** - Filter cepat berdasarkan kriteria

In [141]:
# ============================================================
# 🧠 SMART TOOLS - FUNGSI PENCARIAN & FILTER LANJUTAN
# ============================================================

# -------------------- 1. SIMILAR NAMES FINDER --------------------
def cari_nama_mirip(nama, sumber='SEMUA', top_n=5, show_all_scores=False):
    """
    Temukan top N nama paling mirip dari database Certiport.
    Berguna untuk menemukan variasi penulisan nama.
    
    Parameter:
    - nama: nama yang ingin dicari
    - sumber: 'MCF', 'MOS', atau 'SEMUA' (default)
    - top_n: jumlah kandidat teratas (default 5)
    - show_all_scores: tampilkan semua detail skor
    
    Contoh: cari_nama_mirip('JOHN DOE', 'SEMUA', 10)
    """
    nama_normalized = normalize_name(nama)
    results = []
    
    # Tentukan database yang akan dicari
    databases = []
    if sumber in ['MCF', 'SEMUA']:
        databases.append(('MCF', certiport_mcf_names, certiport_mcf))
    if sumber in ['MOS', 'SEMUA']:
        databases.append(('MOS', certiport_mos_names, certiport_mos))
    
    for db_name, names_list, df in databases:
        for i, cert_name in enumerate(names_list):
            if not cert_name.strip():
                continue
            
            # Hitung semua skor
            score_ratio = fuzz.ratio(nama_normalized, cert_name)
            score_token_set = fuzz.token_set_ratio(nama_normalized, cert_name)
            score_token_sort = fuzz.token_sort_ratio(nama_normalized, cert_name)
            score_partial = fuzz.partial_ratio(nama_normalized, cert_name)
            
            # Hitung kata yang cocok
            matching_words, peserta_words, cert_words = count_matching_words(nama_normalized, cert_name)
            
            max_score = max(score_ratio, score_token_set, score_token_sort, score_partial)
            
            # Ambil info tambahan dari certiport
            try:
                exam_info = df.iloc[i]['Exam'] if 'Exam' in df.columns else '-'
                result_info = df.iloc[i]['Result'] if 'Result' in df.columns else '-'
            except:
                exam_info = '-'
                result_info = '-'
            
            results.append({
                'Database': db_name,
                'Nama': cert_name,
                'Skor Max': max_score,
                'Kata Cocok': f"{matching_words}/{min(peserta_words, cert_words)}",
                'Ratio': score_ratio,
                'TokenSet': score_token_set,
                'TokenSort': score_token_sort,
                'Partial': score_partial,
                'Exam': exam_info,
                'Result': result_info
            })
    
    # Sort by max score
    results = sorted(results, key=lambda x: x['Skor Max'], reverse=True)[:top_n]
    
    print(f"\n{'='*80}")
    print(f"🔍 TOP {top_n} NAMA MIRIP UNTUK: {nama}")
    print(f"   Sumber: {sumber}")
    print(f"{'='*80}")
    
    if not results:
        print("   ❌ Tidak ada hasil ditemukan")
        return pd.DataFrame()
    
    for i, r in enumerate(results, 1):
        status_icon = '✅' if r['Skor Max'] >= 90 else ('🟡' if r['Skor Max'] >= 80 else '⚪')
        result_icon = '✓ Pass' if r['Result'] == 'Pass' else ('✗ Fail' if r['Result'] == 'Fail' else '-')
        
        print(f"\n   {i}. {status_icon} {r['Nama']}")
        print(f"      Database: {r['Database']} | Exam: {r['Exam']} | {result_icon}")
        print(f"      Skor: {r['Skor Max']} | Kata Cocok: {r['Kata Cocok']}")
        
        if show_all_scores:
            print(f"      [Ratio:{r['Ratio']} TokenSet:{r['TokenSet']} TokenSort:{r['TokenSort']} Partial:{r['Partial']}]")
    
    return pd.DataFrame(results)


# -------------------- 2. CROSS-DATABASE SEARCH --------------------
def cari_semua_database(nama, threshold=85):
    """
    Cari nama di SEMUA database (MCF dan MOS) sekaligus.
    Berguna untuk memastikan peserta tidak pernah ujian di program APAPUN.
    
    Parameter:
    - nama: nama yang ingin dicari
    - threshold: skor minimum untuk dianggap match (default 85)
    
    Contoh: cari_semua_database('JOHN DOE')
    """
    print(f"\n{'='*80}")
    print(f"🔎 CROSS-DATABASE SEARCH: {nama}")
    print(f"{'='*80}")
    
    nama_normalized = normalize_name(nama)
    found_anywhere = False
    
    # Cek MCF
    print(f"\n📋 Hasil di MCF (Azure AI-900):")
    print(f"   {'-'*60}")
    match_mcf, score_mcf, type_mcf = find_best_match(nama, certiport_mcf_names, threshold)
    if match_mcf:
        found_anywhere = True
        print(f"   ✅ DITEMUKAN: {match_mcf}")
        print(f"   Skor: {score_mcf} | Tipe: {type_mcf}")
        # Cari detail di dataframe
        idx = certiport_mcf_names.index(match_mcf) if match_mcf in certiport_mcf_names else -1
        if idx >= 0:
            row = certiport_mcf.iloc[idx]
            print(f"   Exam: {row.get('Exam', '-')} | Result: {row.get('Result', '-')}")
    else:
        print(f"   ❌ Tidak ditemukan (skor tertinggi: {score_mcf})")
    
    # Cek MOS
    print(f"\n📋 Hasil di MOS (Office 2019):")
    print(f"   {'-'*60}")
    match_mos, score_mos, type_mos = find_best_match(nama, certiport_mos_names, threshold)
    if match_mos:
        found_anywhere = True
        print(f"   ✅ DITEMUKAN: {match_mos}")
        print(f"   Skor: {score_mos} | Tipe: {type_mos}")
        idx = certiport_mos_names.index(match_mos) if match_mos in certiport_mos_names else -1
        if idx >= 0:
            row = certiport_mos.iloc[idx]
            print(f"   Exam: {row.get('Exam', '-')} | Result: {row.get('Result', '-')}")
    else:
        print(f"   ❌ Tidak ditemukan (skor tertinggi: {score_mos})")
    
    # Summary
    print(f"\n{'='*80}")
    if found_anywhere:
        print(f"⚠️  KESIMPULAN: Peserta SUDAH PERNAH ujian sertifikasi!")
        if match_mcf:
            print(f"   → Ditemukan di MCF dengan skor {score_mcf}")
        if match_mos:
            print(f"   → Ditemukan di MOS dengan skor {score_mos}")
    else:
        print(f"✅ KESIMPULAN: Peserta BELUM PERNAH ujian di database manapun")
    print(f"{'='*80}")
    
    return {
        'MCF': {'match': match_mcf, 'score': score_mcf},
        'MOS': {'match': match_mos, 'score': score_mos},
        'found_anywhere': found_anywhere
    }


# -------------------- 3. BATCH LOOKUP TOOL --------------------
def cek_batch(list_nama, sumber='AUTO'):
    """
    Cek banyak nama sekaligus dan tampilkan hasilnya dalam tabel.
    
    Parameter:
    - list_nama: list of tuples [(nama, program), ...] atau list of strings
    - sumber: 'MCF', 'MOS', 'AUTO' (otomatis berdasarkan database), atau 'SEMUA'
    
    Contoh 1: cek_batch(['JOHN DOE', 'JANE SMITH'], 'MOS')
    Contoh 2: cek_batch([('JOHN DOE', 'MCF'), ('JANE SMITH', 'MOS')])
    """
    print(f"\n{'='*80}")
    print(f"📋 BATCH LOOKUP - Mengecek {len(list_nama)} nama")
    print(f"{'='*80}")
    
    results = []
    
    for item in list_nama:
        # Parse input
        if isinstance(item, tuple):
            nama, program = item
        else:
            nama = item
            program = sumber if sumber != 'AUTO' else 'SEMUA'
        
        # Tentukan database
        if program == 'SEMUA' or sumber == 'SEMUA':
            # Cek kedua database
            res = cari_semua_database(nama, threshold=85)
            if res['MCF']['match']:
                results.append({
                    'Input': nama,
                    'Program': 'MCF',
                    'Status': '❌ SUDAH UJIAN',
                    'Match': res['MCF']['match'],
                    'Skor': res['MCF']['score']
                })
            elif res['MOS']['match']:
                results.append({
                    'Input': nama,
                    'Program': 'MOS', 
                    'Status': '❌ SUDAH UJIAN',
                    'Match': res['MOS']['match'],
                    'Skor': res['MOS']['score']
                })
            else:
                results.append({
                    'Input': nama,
                    'Program': 'SEMUA',
                    'Status': '✅ BELUM UJIAN',
                    'Match': '-',
                    'Skor': max(res['MCF']['score'], res['MOS']['score'])
                })
        else:
            # Cek database spesifik
            names_to_search = certiport_mcf_names if program == 'MCF' else certiport_mos_names
            match, score, _ = find_best_match(nama, names_to_search, threshold=85)
            
            results.append({
                'Input': nama,
                'Program': program,
                'Status': '❌ SUDAH UJIAN' if match else '✅ BELUM UJIAN',
                'Match': match if match else '-',
                'Skor': score
            })
    
    # Tampilkan hasil
    df_results = pd.DataFrame(results)
    print(f"\n{df_results.to_string(index=False)}")
    
    # Summary
    sudah_ujian = len([r for r in results if '❌' in r['Status']])
    belum_ujian = len([r for r in results if '✅' in r['Status']])
    
    print(f"\n{'='*80}")
    print(f"📊 RINGKASAN BATCH:")
    print(f"   ✅ Belum Ujian (APPROVE)      : {belum_ujian} orang")
    print(f"   ❌ Sudah Ujian (JANGAN APPROVE): {sudah_ujian} orang")
    print(f"{'='*80}")
    
    return df_results


print("✅ Smart Tools - Similar Names Finder, Cross-DB Search, Batch Lookup berhasil dibuat!")
print("\n📖 CARA PAKAI:")
print("="*60)
print("1️⃣  cari_nama_mirip('NAMA', 'SEMUA', top_n=10)")
print("    → Tampilkan 10 nama paling mirip dari semua database")
print("")
print("2️⃣  cari_semua_database('NAMA')")
print("    → Cek nama di MCF dan MOS sekaligus")
print("")
print("3️⃣  cek_batch(['NAMA1', 'NAMA2', 'NAMA3'], 'MOS')")
print("    → Cek banyak nama sekaligus")
print("="*60)

✅ Smart Tools - Similar Names Finder, Cross-DB Search, Batch Lookup berhasil dibuat!

📖 CARA PAKAI:
1️⃣  cari_nama_mirip('NAMA', 'SEMUA', top_n=10)
    → Tampilkan 10 nama paling mirip dari semua database

2️⃣  cari_semua_database('NAMA')
    → Cek nama di MCF dan MOS sekaligus

3️⃣  cek_batch(['NAMA1', 'NAMA2', 'NAMA3'], 'MOS')
    → Cek banyak nama sekaligus


In [142]:
# ============================================================
# 🧠 SMART TOOLS - PENCARIAN MULTI-KRITERIA & QUICK FILTERS
# ============================================================

# -------------------- 4. MULTI-CRITERIA SEARCH --------------------
def cari_peserta(nama=None, nim=None, jurusan=None, program=None, 
                 skor_min=None, skor_max=None, status=None, source_df='all'):
    """
    Cari peserta berdasarkan berbagai kriteria sekaligus.
    
    Parameter:
    - nama: filter berdasarkan nama (partial match)
    - nim: filter berdasarkan NIM (partial match)
    - jurusan: filter berdasarkan jurusan (partial match)
    - program: 'MCF' atau 'MOS'
    - skor_min: skor kecocokan minimum
    - skor_max: skor kecocokan maximum
    - status: 'DITEMUKAN' atau 'TIDAK DITEMUKAN'
    - source_df: 'hasil' (hasil crosscheck), 'approved', 'all'
    
    Contoh: cari_peserta(jurusan='Informatika', skor_min=85, status='DITEMUKAN')
    """
    # Pilih source dataframe
    if source_df == 'hasil':
        df = results_df.copy()
    elif source_df == 'approved':
        df = approved_filter_df.copy()
    else:
        df = results_df.copy()
    
    # Apply filters
    if nama:
        df = df[df['Nama Peserta'].str.upper().str.contains(nama.upper(), na=False)]
    
    if nim:
        df = df[df['NIM'].astype(str).str.contains(str(nim), na=False)]
    
    if jurusan:
        df = df[df['Jurusan'].str.upper().str.contains(jurusan.upper(), na=False)]
    
    if program:
        df = df[df['Sumber Data'] == program.upper()]
    
    if skor_min is not None:
        df = df[df['Skor Kecocokan'] >= skor_min]
    
    if skor_max is not None:
        df = df[df['Skor Kecocokan'] <= skor_max]
    
    if status:
        if 'DITEMUKAN' in status.upper() and 'TIDAK' not in status.upper():
            df = df[df['Status Certiport'].str.contains('✅', na=False)]
        elif 'TIDAK' in status.upper():
            df = df[df['Status Certiport'].str.contains('❌', na=False)]
    
    # Display results
    print(f"\n{'='*80}")
    print(f"🔍 HASIL PENCARIAN MULTI-KRITERIA")
    print(f"{'='*80}")
    print(f"Filter: ", end="")
    filters_used = []
    if nama: filters_used.append(f"nama='{nama}'")
    if nim: filters_used.append(f"nim='{nim}'")
    if jurusan: filters_used.append(f"jurusan='{jurusan}'")
    if program: filters_used.append(f"program='{program}'")
    if skor_min: filters_used.append(f"skor>={skor_min}")
    if skor_max: filters_used.append(f"skor<={skor_max}")
    if status: filters_used.append(f"status='{status}'")
    print(", ".join(filters_used) if filters_used else "Tidak ada filter")
    print(f"\nDitemukan: {len(df)} peserta")
    print(f"{'='*80}")
    
    if len(df) > 0:
        cols = ['Nama Peserta', 'NIM', 'Jurusan', 'Sumber Data', 'Skor Kecocokan', 'Status Certiport']
        cols = [c for c in cols if c in df.columns]
        print(df[cols].to_string(index=False))
    
    return df


# -------------------- 5. QUICK FILTER FUNCTIONS --------------------
def filter_by_jurusan(jurusan_keyword):
    """Filter peserta berdasarkan jurusan"""
    return cari_peserta(jurusan=jurusan_keyword)

def filter_by_score_range(min_score, max_score):
    """Filter peserta berdasarkan rentang skor kecocokan"""
    return cari_peserta(skor_min=min_score, skor_max=max_score)

def filter_borderline(threshold_low=85, threshold_high=94):
    """
    Tampilkan kasus BORDERLINE yang perlu review manual.
    Ini adalah kasus dengan skor antara threshold_low dan threshold_high.
    """
    print(f"\n{'='*80}")
    print(f"🟡 KASUS BORDERLINE (Skor {threshold_low}-{threshold_high}) - PERLU CEK MANUAL")
    print(f"{'='*80}")
    
    borderline = results_df[
        (results_df['Skor Kecocokan'] >= threshold_low) & 
        (results_df['Skor Kecocokan'] <= threshold_high)
    ].copy()
    
    if len(borderline) > 0:
        print(f"\nDitemukan {len(borderline)} kasus borderline:\n")
        cols = ['Nama Peserta', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan', 'Status Certiport']
        print(borderline[cols].to_string(index=False))
        
        # Breakdown
        print(f"\n📊 Breakdown:")
        print(f"   Skor 85-89: {len(borderline[borderline['Skor Kecocokan'] < 90])} peserta")
        print(f"   Skor 90-94: {len(borderline[borderline['Skor Kecocokan'] >= 90])} peserta")
    else:
        print(f"\n✅ Tidak ada kasus borderline!")
    
    return borderline


def filter_high_confidence(min_score=95):
    """Filter peserta dengan skor kecocokan tinggi (high confidence match)"""
    print(f"\n{'='*80}")
    print(f"🔴 HIGH CONFIDENCE MATCH (Skor ≥{min_score}) - PASTI SUDAH UJIAN")
    print(f"{'='*80}")
    
    high_conf = results_df[
        (results_df['Skor Kecocokan'] >= min_score) & 
        (results_df['Status Certiport'].str.contains('✅', na=False))
    ].copy()
    
    if len(high_conf) > 0:
        print(f"\nDitemukan {len(high_conf)} peserta dengan match tinggi:\n")
        cols = ['Nama Peserta', 'Sumber Data', 'Nama di Certiport', 'Skor Kecocokan']
        print(high_conf[cols].to_string(index=False))
    else:
        print(f"\n✅ Tidak ada match dengan skor ≥{min_score}")
    
    return high_conf


# -------------------- 6. STATISTIK DISTRIBUSI SKOR --------------------
def analisis_distribusi_skor():
    """
    Analisis distribusi skor kecocokan untuk optimasi threshold.
    Membantu memahami kualitas matching dan menentukan threshold optimal.
    """
    print(f"\n{'='*80}")
    print(f"📊 ANALISIS DISTRIBUSI SKOR KECOCOKAN")
    print(f"{'='*80}")
    
    scores = results_df['Skor Kecocokan'].dropna()
    
    print(f"\n📈 STATISTIK DASAR:")
    print(f"   Total Data      : {len(scores)}")
    print(f"   Rata-rata       : {scores.mean():.1f}")
    print(f"   Median          : {scores.median():.1f}")
    print(f"   Minimum         : {scores.min():.0f}")
    print(f"   Maximum         : {scores.max():.0f}")
    print(f"   Std Deviation   : {scores.std():.1f}")
    
    print(f"\n📊 DISTRIBUSI PER RANGE:")
    ranges = [
        (0, 49, '⚪ Sangat Rendah'),
        (50, 69, '⚪ Rendah'),
        (70, 79, '🟡 Sedang'),
        (80, 84, '🟡 Cukup'),
        (85, 89, '🟠 Borderline Low'),
        (90, 94, '🟠 Borderline High'),
        (95, 99, '🔴 Tinggi'),
        (100, 100, '🔴 Exact Match')
    ]
    
    for low, high, label in ranges:
        count = len(scores[(scores >= low) & (scores <= high)])
        pct = count/len(scores)*100 if len(scores) > 0 else 0
        bar = '█' * int(pct/2)
        print(f"   {label:20} ({low:3}-{high:3}): {count:4} ({pct:5.1f}%) {bar}")
    
    # Analisis threshold
    print(f"\n🎯 ANALISIS THRESHOLD:")
    for threshold in [80, 85, 90, 95]:
        above = len(scores[scores >= threshold])
        below = len(scores[scores < threshold])
        print(f"   Threshold {threshold}: {above} match (≥{threshold}) | {below} tidak match (<{threshold})")
    
    return scores


print("✅ Smart Tools - Multi-Criteria Search & Quick Filters berhasil dibuat!")
print("\n📖 CARA PAKAI:")
print("="*60)
print("4️⃣  cari_peserta(jurusan='Informatika', skor_min=85)")
print("    → Cari peserta Informatika dengan skor ≥85")
print("")
print("5️⃣  filter_borderline(85, 94)")
print("    → Tampilkan kasus borderline yang perlu review manual")
print("")
print("6️⃣  filter_high_confidence(95)")
print("    → Tampilkan match dengan confidence tinggi")
print("")
print("7️⃣  analisis_distribusi_skor()")
print("    → Lihat distribusi skor untuk optimasi threshold")
print("="*60)

✅ Smart Tools - Multi-Criteria Search & Quick Filters berhasil dibuat!

📖 CARA PAKAI:
4️⃣  cari_peserta(jurusan='Informatika', skor_min=85)
    → Cari peserta Informatika dengan skor ≥85

5️⃣  filter_borderline(85, 94)
    → Tampilkan kasus borderline yang perlu review manual

6️⃣  filter_high_confidence(95)
    → Tampilkan match dengan confidence tinggi

7️⃣  analisis_distribusi_skor()
    → Lihat distribusi skor untuk optimasi threshold


In [143]:
# ============================================================
# 🧠 SMART TOOLS - DATA VALIDATION & QUALITY CHECK
# ============================================================

# -------------------- 7. DATA VALIDATION DASHBOARD --------------------
def validasi_data():
    """
    Cek kualitas data dan tampilkan potensi masalah:
    - NIM duplikat
    - Nama duplikat
    - Format nama aneh
    - Data tidak lengkap
    """
    print(f"\n{'='*80}")
    print(f"🔍 DATA VALIDATION DASHBOARD")
    print(f"{'='*80}")
    
    issues = []
    
    # 1. Cek NIM duplikat
    print(f"\n1️⃣  CEK NIM DUPLIKAT:")
    nim_duplicates = peserta_df[peserta_df.duplicated(subset=['NIM'], keep=False)].copy()
    if len(nim_duplicates) > 0:
        nim_groups = nim_duplicates.groupby('NIM').size().reset_index(name='count')
        nim_groups = nim_groups[nim_groups['count'] > 1]
        print(f"   ⚠️  Ditemukan {len(nim_groups)} NIM duplikat!")
        for _, row in nim_groups.iterrows():
            dupes = peserta_df[peserta_df['NIM'] == row['NIM']]
            print(f"      NIM {row['NIM']}: {row['count']}x")
            for _, d in dupes.iterrows():
                print(f"         - {d['Nama']} ({d['Sumber']})")
            issues.append(f"NIM duplikat: {row['NIM']}")
    else:
        print(f"   ✅ Tidak ada NIM duplikat")
    
    # 2. Cek nama duplikat (beda NIM)
    print(f"\n2️⃣  CEK NAMA DUPLIKAT:")
    peserta_df['Nama_Upper'] = peserta_df['Nama'].str.upper().str.strip()
    name_duplicates = peserta_df[peserta_df.duplicated(subset=['Nama_Upper'], keep=False)].copy()
    if len(name_duplicates) > 0:
        name_groups = name_duplicates.groupby('Nama_Upper').size().reset_index(name='count')
        name_groups = name_groups[name_groups['count'] > 1]
        unique_nims = name_duplicates.groupby('Nama_Upper')['NIM'].nunique()
        multi_nim = unique_nims[unique_nims > 1]
        
        if len(multi_nim) > 0:
            print(f"   ⚠️  Ditemukan {len(multi_nim)} nama sama dengan NIM berbeda!")
            for nama in multi_nim.index:
                dupes = peserta_df[peserta_df['Nama_Upper'] == nama]
                print(f"      {nama}:")
                for _, d in dupes.iterrows():
                    print(f"         - NIM: {d['NIM']} ({d['Sumber']})")
                issues.append(f"Nama duplikat dengan NIM berbeda: {nama}")
        else:
            print(f"   ✅ Tidak ada nama duplikat yang mencurigakan")
    else:
        print(f"   ✅ Tidak ada nama duplikat")
    
    # 3. Cek format nama aneh
    print(f"\n3️⃣  CEK FORMAT NAMA:")
    strange_names = []
    for _, row in peserta_df.iterrows():
        nama = str(row['Nama'])
        # Cek nama terlalu pendek
        if len(nama.replace(' ', '')) < 4:
            strange_names.append((row['Nama'], row['NIM'], 'Terlalu pendek'))
        # Cek nama hanya 1 kata
        elif len(nama.split()) == 1 and '-' not in nama and '.' not in nama:
            strange_names.append((row['Nama'], row['NIM'], 'Hanya 1 kata'))
        # Cek nama dengan angka
        elif any(c.isdigit() for c in nama):
            strange_names.append((row['Nama'], row['NIM'], 'Mengandung angka'))
    
    if strange_names:
        print(f"   ⚠️  Ditemukan {len(strange_names)} nama dengan format aneh:")
        for nama, nim, reason in strange_names[:10]:  # Limit 10
            print(f"      - {nama} (NIM: {nim}) → {reason}")
            issues.append(f"Format nama aneh: {nama}")
        if len(strange_names) > 10:
            print(f"      ... dan {len(strange_names) - 10} lainnya")
    else:
        print(f"   ✅ Semua format nama terlihat normal")
    
    # 4. Cek data kosong
    print(f"\n4️⃣  CEK DATA KOSONG:")
    empty_check = {
        'Nama': peserta_df['Nama'].isna().sum(),
        'NIM': peserta_df['NIM'].isna().sum(),
        'Jurusan': peserta_df['Jurusan'].isna().sum(),
    }
    has_empty = False
    for col, count in empty_check.items():
        if count > 0:
            print(f"   ⚠️  {col} kosong: {count} baris")
            issues.append(f"{col} kosong: {count}")
            has_empty = True
    if not has_empty:
        print(f"   ✅ Tidak ada data kosong")
    
    # 5. Summary
    print(f"\n{'='*80}")
    print(f"📊 SUMMARY VALIDASI DATA")
    print(f"{'='*80}")
    if issues:
        print(f"   ⚠️  Total {len(issues)} potensi masalah ditemukan")
        print(f"   Disarankan untuk mereview data sebelum proses lebih lanjut")
    else:
        print(f"   ✅ Data tervalidasi dengan baik!")
    
    # Cleanup
    if 'Nama_Upper' in peserta_df.columns:
        peserta_df.drop('Nama_Upper', axis=1, inplace=True)
    
    return issues


# -------------------- 8. SMART AUTO-REVIEW --------------------
def auto_review():
    """
    Otomatis review semua hasil dan berikan rekomendasi final.
    Mengurangi kebutuhan pengecekan manual.
    """
    print(f"\n{'='*80}")
    print(f"🤖 AUTO-REVIEW - ANALISIS OTOMATIS SEMUA HASIL")
    print(f"{'='*80}")
    
    review_results = {
        'confident_approve': [],
        'confident_reject': [],
        'need_review': [],
        'cross_db_warning': []
    }
    
    print(f"\nMenganalisis {len(results_df)} peserta NOT APPROVED...")
    
    for idx, row in results_df.iterrows():
        nama = row['Nama Peserta']
        skor = row['Skor Kecocokan']
        status = row['Status Certiport']
        sumber = row['Sumber Data']
        match = row['Nama di Certiport']
        
        # Case 1: High confidence - SUDAH UJIAN (skor >= 95)
        if '✅' in status and skor >= 95:
            review_results['confident_reject'].append({
                'Nama': nama,
                'Skor': skor,
                'Match': match,
                'Reason': f'Skor tinggi ({skor}%), pasti match'
            })
        
        # Case 2: Confident - BELUM UJIAN (tidak ditemukan dan skor < 80)
        elif '❌' in status and skor < 80:
            # Double check di database lain
            other_db = certiport_mos_names if sumber == 'MCF' else certiport_mcf_names
            other_match, other_score, _ = find_best_match(nama, other_db, threshold=85)
            
            if other_match:
                review_results['cross_db_warning'].append({
                    'Nama': nama,
                    'Program Daftar': sumber,
                    'Ditemukan Di': 'MOS' if sumber == 'MCF' else 'MCF',
                    'Match': other_match,
                    'Skor': other_score,
                    'Reason': 'Tidak di program ini, tapi ada di program lain!'
                })
            else:
                review_results['confident_approve'].append({
                    'Nama': nama,
                    'Skor': skor,
                    'Reason': f'Skor rendah ({skor}%), tidak ada di semua DB'
                })
        
        # Case 3: Borderline - PERLU REVIEW (skor 80-94 atau status ambigu)
        elif '✅' in status and 80 <= skor < 95:
            review_results['need_review'].append({
                'Nama': nama,
                'Skor': skor,
                'Match': match,
                'Reason': f'Skor borderline ({skor}%), perlu verifikasi manual'
            })
        
        # Case 4: Low score but found
        elif '❌' in status and skor >= 80:
            review_results['need_review'].append({
                'Nama': nama,
                'Skor': skor,
                'Match': match if match else '-',
                'Reason': f'Skor cukup tinggi ({skor}%) tapi tidak match threshold'
            })
    
    # Display results
    print(f"\n" + "="*80)
    print(f"✅ CONFIDENT APPROVE ({len(review_results['confident_approve'])} peserta)")
    print("="*80)
    if review_results['confident_approve']:
        df_approve = pd.DataFrame(review_results['confident_approve'])
        print(df_approve.to_string(index=False))
    else:
        print("   Tidak ada")
    
    print(f"\n" + "="*80)
    print(f"❌ CONFIDENT REJECT ({len(review_results['confident_reject'])} peserta)")
    print("="*80)
    if review_results['confident_reject']:
        df_reject = pd.DataFrame(review_results['confident_reject'])
        print(df_reject.to_string(index=False))
    else:
        print("   Tidak ada")
    
    print(f"\n" + "="*80)
    print(f"⚠️  CROSS-DATABASE WARNING ({len(review_results['cross_db_warning'])} peserta)")
    print("   Peserta ini tidak ditemukan di program yang didaftar,")
    print("   TAPI ditemukan di program lain!")
    print("="*80)
    if review_results['cross_db_warning']:
        df_cross = pd.DataFrame(review_results['cross_db_warning'])
        print(df_cross.to_string(index=False))
    else:
        print("   Tidak ada")
    
    print(f"\n" + "="*80)
    print(f"🟡 PERLU REVIEW MANUAL ({len(review_results['need_review'])} peserta)")
    print("="*80)
    if review_results['need_review']:
        df_review = pd.DataFrame(review_results['need_review'])
        print(df_review.to_string(index=False))
    else:
        print("   Tidak ada")
    
    # Final Summary
    total_confident = len(review_results['confident_approve']) + len(review_results['confident_reject'])
    total_need_review = len(review_results['need_review']) + len(review_results['cross_db_warning'])
    
    print(f"\n" + "="*80)
    print(f"📊 RINGKASAN AUTO-REVIEW")
    print("="*80)
    print(f"   ✅ Confident Decisions: {total_confident} peserta ({total_confident/len(results_df)*100:.1f}%)")
    print(f"   🟡 Need Manual Review : {total_need_review} peserta ({total_need_review/len(results_df)*100:.1f}%)")
    print(f"\n   🎯 Anda hanya perlu review {total_need_review} dari {len(results_df)} peserta!")
    
    return review_results


print("✅ Smart Tools - Data Validation & Auto-Review berhasil dibuat!")
print("\n📖 CARA PAKAI:")
print("="*60)
print("8️⃣  validasi_data()")
print("    → Cek kualitas data (duplikat, format aneh, dll)")
print("")
print("9️⃣  auto_review()")
print("    → Auto-review semua hasil dengan rekomendasi")
print("    → Cross-check database untuk deteksi lebih akurat")
print("="*60)

✅ Smart Tools - Data Validation & Auto-Review berhasil dibuat!

📖 CARA PAKAI:
8️⃣  validasi_data()
    → Cek kualitas data (duplikat, format aneh, dll)

9️⃣  auto_review()
    → Auto-review semua hasil dengan rekomendasi
    → Cross-check database untuk deteksi lebih akurat


In [144]:
# ============================================================
# 🧠 SMART TOOLS - EXPORT & UTILITY FUNCTIONS
# ============================================================

# -------------------- 9. SMART EXPORT FILTERED DATA --------------------
def export_filtered(filter_func, output_file, **kwargs):
    """
    Export data yang sudah difilter ke file CSV/Excel.
    
    Parameter:
    - filter_func: fungsi filter (cari_peserta, filter_borderline, etc)
    - output_file: nama file output (dengan ekstensi .csv atau .xlsx)
    - **kwargs: parameter untuk filter_func
    
    Contoh: export_filtered(cari_peserta, 'informatika.csv', jurusan='Informatika')
    """
    # Apply filter
    df = filter_func(**kwargs)
    
    if len(df) == 0:
        print("⚠️  Tidak ada data untuk di-export")
        return
    
    # Export berdasarkan ekstensi
    if output_file.endswith('.xlsx'):
        df.to_excel(output_file, index=False)
    else:
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ Data berhasil di-export ke: {output_file}")
    print(f"   Total {len(df)} baris")


# -------------------- 10. QUICK STATUS CHECK --------------------
def status_cepat(nama):
    """
    Cek status nama dengan cepat - satu baris output.
    
    Contoh: status_cepat('JOHN DOE')
    """
    # Cek di hasil crosscheck
    match_df = results_df[results_df['Nama Peserta'].str.upper().str.contains(nama.upper(), na=False)]
    
    if len(match_df) > 0:
        for _, row in match_df.iterrows():
            status = '❌ JANGAN APPROVE' if '✅' in row['Status Certiport'] else '✅ APPROVE'
            print(f"{row['Nama Peserta']} ({row['Sumber Data']}): {status} | Skor: {row['Skor Kecocokan']}")
    else:
        # Cek di approved
        match_approved = approved_df[approved_df['Nama'].str.upper().str.contains(nama.upper(), na=False)]
        if len(match_approved) > 0:
            for _, row in match_approved.iterrows():
                print(f"{row['Nama']} ({row['Sumber']}): Status APPROVED")
        else:
            print(f"'{nama}' tidak ditemukan di data peserta")


# -------------------- 11. HELP COMMAND --------------------
def bantuan():
    """Tampilkan daftar semua fungsi smart tools yang tersedia"""
    print(f"""
{'='*80}
🧠 SMART TOOLS - PANDUAN LENGKAP
{'='*80}

📌 FUNGSI PENCARIAN:
    cari_nama(nama, sumber)
        → Cari nama di database Certiport
        → Contoh: cari_nama('JOHN DOE', 'MOS')
    
    cari_nama_mirip(nama, sumber, top_n)
        → Tampilkan top N nama paling mirip
        → Contoh: cari_nama_mirip('JOHN', 'SEMUA', 10)
    
    cari_semua_database(nama)
        → Cek nama di MCF dan MOS sekaligus
        → Contoh: cari_semua_database('JOHN DOE')
    
    cek_batch(list_nama, sumber)
        → Cek banyak nama sekaligus
        → Contoh: cek_batch(['NAMA1', 'NAMA2'], 'MOS')

📌 FUNGSI FILTER:
    cari_peserta(nama, nim, jurusan, program, skor_min, skor_max, status)
        → Cari dengan multi-kriteria
        → Contoh: cari_peserta(jurusan='Informatika', skor_min=90)
    
    filter_borderline(threshold_low, threshold_high)
        → Tampilkan kasus borderline
        → Contoh: filter_borderline(85, 94)
    
    filter_high_confidence(min_score)
        → Tampilkan match dengan confidence tinggi
        → Contoh: filter_high_confidence(95)

📌 FUNGSI ANALISIS:
    analisis_distribusi_skor()
        → Lihat distribusi skor kecocokan
    
    validasi_data()
        → Cek kualitas data (duplikat, format, dll)
    
    auto_review()
        → Auto-review dengan cross-database check
        → Mengidentifikasi peserta di program lain

📌 FUNGSI UTILITY:
    status_cepat(nama)
        → Quick check status satu nama
        → Contoh: status_cepat('JOHN')
    
    export_filtered(filter_func, output_file, **kwargs)
        → Export hasil filter ke file

{'='*80}
""")


print("✅ Smart Tools - Export & Utility Functions berhasil dibuat!")
print("\n💡 TIP: Ketik bantuan() untuk melihat panduan lengkap semua fungsi!")
print("="*60)

✅ Smart Tools - Export & Utility Functions berhasil dibuat!

💡 TIP: Ketik bantuan() untuk melihat panduan lengkap semua fungsi!


In [145]:
# ============================================================
# FUNGSI PENCARIAN DASAR (IMPROVED)
# ============================================================

def cari_nama(nama, sumber='MOS', show_detail=True):
    """
    Fungsi untuk mencari nama secara manual di database Certiport
    
    Parameter:
    - nama: nama yang ingin dicari
    - sumber: 'MCF', 'MOS', atau 'SEMUA'
    - show_detail: tampilkan detail lengkap atau hanya hasil
    
    Contoh: 
        cari_nama('JOHN DOE', 'MOS')
        cari_nama('JOHN DOE', 'SEMUA')  # Cek di kedua database
    """
    if sumber == 'SEMUA':
        return cari_semua_database(nama)
    
    if sumber == 'MCF':
        names_to_search = certiport_mcf_names
        df_source = certiport_mcf
    else:
        names_to_search = certiport_mos_names
        df_source = certiport_mos
    
    match, score, match_type = find_best_match(nama, names_to_search)
    
    print(f"\n🔍 Hasil Pencarian untuk: {nama}")
    print(f"   Sumber: {sumber}")
    print(f"   {'='*50}")
    
    if match:
        print(f"   ✅ DITEMUKAN!")
        print(f"   Nama di Certiport: {match}")
        print(f"   Skor Kecocokan: {score}")
        print(f"   Tipe Match: {match_type}")
        
        if show_detail:
            # Tampilkan detail dari certiport
            idx = names_to_search.index(match) if match in names_to_search else -1
            if idx >= 0:
                row = df_source.iloc[idx]
                print(f"\n   📋 Detail Certiport:")
                print(f"      Exam: {row.get('Exam', '-')}")
                print(f"      Result: {row.get('Result', '-')}")
                print(f"      Score: {row.get('Score', '-')}")
    else:
        print(f"   ❌ TIDAK DITEMUKAN")
        print(f"   Skor Tertinggi: {score}")
        print(f"\n   💡 TIP: Coba cari_nama_mirip('{nama}', '{sumber}', 5)")
        print(f"           untuk melihat kandidat paling mirip")
    
    return {'match': match, 'score': score, 'type': match_type}


print("✅ Fungsi cari_nama() (IMPROVED) siap digunakan!")
print("\n📖 Cara Pakai:")
print("   cari_nama('NAMA LENGKAP', 'MOS')   # Untuk MOS")
print("   cari_nama('NAMA LENGKAP', 'MCF')   # Untuk MCF")
print("   cari_nama('NAMA LENGKAP', 'SEMUA') # Cek kedua database")

✅ Fungsi cari_nama() (IMPROVED) siap digunakan!

📖 Cara Pakai:
   cari_nama('NAMA LENGKAP', 'MOS')   # Untuk MOS
   cari_nama('NAMA LENGKAP', 'MCF')   # Untuk MCF
   cari_nama('NAMA LENGKAP', 'SEMUA') # Cek kedua database
